# log-samples-eval-callback — worked example 2: Skip a warm-up window then log on cadence

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-samples-eval-callback`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

An offset-aligned callback gates on `step >= start_step` (a warm-up skip) and then fires when `(step - start_step) % eval_every == 0`. This avoids logging noise from an un-warmed model and aligns the cadence to the first post-warmup step.

## Worked solution

We add a warm-up offset to the cadence logic.

1. **Warm-up gate.** While `step < start_step` we `continue` — no logging during warm-up.
2. **Offset-aligned cadence.** Once past the gate, `(step - start_step) % eval_every == 0` fires at `start_step`, `start_step + K`, etc. Subtracting `start_step` re-zeroes the cadence to the first eligible step.
3. **Fire body.** Append the sample record and count the fire.
4. **Edge case.** If `start_step >= n_steps` nothing fires.

The demo runs 12 steps with `start_step=4`, `eval_every=3`, and prints that it fired at 4, 7, 10.

In [ ]:
def run_with_offset(n_steps, eval_every, start_step, n_eval, sink):
    n_fires = 0
    for step in range(n_steps):
        if step < start_step:
            continue
        if (step - start_step) % eval_every != 0:
            continue
        samples = [f'step={step}-sample={i}' for i in range(n_eval)]
        sink.append({'step': step, 'samples': samples})
        n_fires += 1
    return n_fires

sink = []
fires = run_with_offset(n_steps=12, eval_every=3, start_step=4, n_eval=2, sink=sink)
print('fires:', fires)
print('fired at steps:', [d['step'] for d in sink])